# 08 — Generate paper figures

Generate Figure 1 from standardized geometry products and produce the key-event waveform and catalogue-summary figures from finalized CSV and waveform products. This notebook performs visualization only; it does not recompute scientific measurements.


> **Archived pre-cleanup version.** This executed notebook is retained for provenance and is not a production dependency.


# Part I — Figure 1: deployment geometry


## 1. Imports and project paths

The notebook may be run from either the project root or the `notebooks/`
directory.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from pyproj import CRS, Geod, Transformer

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

MODULE_DIR = PROJECT_ROOT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from project_config import ensure_output_dirs
from geometry_products import read_geometry_products
from figure1_utils import (
    get_sensor_location,
    make_figure1,
    save_figure,
)

PATHS = ensure_output_dirs(PROJECT_ROOT)
DERIVED_DIR = PATHS["derived"]
FIGURE_DIR = PATHS["figures"]

WGS84 = CRS.from_epsg(4326)
UTM17N = CRS.from_epsg(32617)

geod = Geod(ellps="WGS84")
ll_to_utm = Transformer.from_crs(
    WGS84,
    UTM17N,
    always_xy=True,
)
utm_to_ll = Transformer.from_crs(
    UTM17N,
    WGS84,
    always_xy=True,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Derived data: {DERIVED_DIR}")
print(f"Figure output: {FIGURE_DIR}")


## 2. Load the standardized geometry products

Notebook 02 writes the event inventory and normalized channel, station, and
KML-location tables. Figure 1 consumes those products directly rather than
rereading StationXML or KML independently.


In [ ]:
(
    inventory_event,
    channels_df,
    stations_df,
    locations_df,
) = read_geometry_products(DERIVED_DIR)

print("Channel columns:")
print(channels_df.columns.tolist())

print("\nLocation columns:")
print(locations_df.columns.tolist())

display(stations_df)
display(channels_df)
display(locations_df)


## 3. Adapt the channel table for Figure 1

The standardized channel table uses descriptive column names such as
`latitude` and `longitude`. The existing Figure 1 utility expects the
concise plotting schema `sensor`, `label`, `lat`, `lon`, `easting`, and
`northing`. This cell creates those derived fields explicitly.

The three seismic components represent one physical seismometer location,
so they are collapsed to a single `Seismometer` row.


In [ ]:
CHANNEL_TO_SENSOR = {
    "DHZ": "Seismometer",
    "DHN": "Seismometer",
    "DHE": "Seismometer",
    "HHZ": "Seismometer",
    "HHN": "Seismometer",
    "HHE": "Seismometer",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
    "HD1": "HD1",
    "HD2": "HD2",
    "HD3": "HD3",
}

CHANNEL_TO_LABEL = {
    "DHZ": "BCHH",
    "DHN": "BCHH",
    "DHE": "BCHH",
    "HHZ": "BCHH",
    "HHN": "BCHH",
    "HHE": "BCHH",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
    "HD1": "HD1",
    "HD2": "HD2",
    "HD3": "HD3",
}

required_channel_columns = {
    "channel",
    "latitude",
    "longitude",
}
missing = required_channel_columns.difference(channels_df.columns)
if missing:
    raise KeyError(
        "The standardized channel table is missing required columns: "
        f"{sorted(missing)}"
    )

channels_for_map = channels_df.copy()
channel_codes = (
    channels_for_map["channel"]
    .astype(str)
    .str.upper()
)

channels_for_map["sensor"] = channel_codes.map(CHANNEL_TO_SENSOR)
channels_for_map["label"] = channel_codes.map(CHANNEL_TO_LABEL)
channels_for_map["lat"] = channels_for_map["latitude"].astype(float)
channels_for_map["lon"] = channels_for_map["longitude"].astype(float)

eastings, northings = ll_to_utm.transform(
    channels_for_map["lon"].to_numpy(),
    channels_for_map["lat"].to_numpy(),
)
channels_for_map["easting"] = eastings
channels_for_map["northing"] = northings

unmapped = channels_for_map.loc[
    channels_for_map["sensor"].isna(),
    ["seed_id", "channel", "sensor_description"],
]
if not unmapped.empty:
    print("Ignoring channels that are not used in Figure 1:")
    display(unmapped)

mapped = channels_for_map.dropna(subset=["sensor"]).copy()

seismometer_rows = mapped.loc[
    mapped["sensor"] == "Seismometer"
]
if seismometer_rows.empty:
    raise ValueError("No seismic component was mapped to 'Seismometer'.")

# All three components are co-located; retain one representative row.
seismometer_row = seismometer_rows.iloc[[0]].copy()

infrasound_rows = (
    mapped.loc[mapped["sensor"].isin(["HD1", "HD2", "HD3"])]
    .sort_values("sensor")
    .drop_duplicates(subset=["sensor"])
)

bchh_sensors_df = pd.concat(
    [seismometer_row, infrasound_rows],
    ignore_index=True,
)

expected_sensors = {"Seismometer", "HD1", "HD2", "HD3"}
actual_sensors = set(bchh_sensors_df["sensor"])
if actual_sensors != expected_sensors:
    raise ValueError(
        "Expected exactly these physical sensors: "
        f"{sorted(expected_sensors)}; found {sorted(actual_sensors)}"
    )

figure1_columns = [
    "sensor",
    "label",
    "lat",
    "lon",
    "easting",
    "northing",
    "channel",
    "seed_id",
    "sensor_description",
]
display(bchh_sensors_df[figure1_columns])


## 4. Extract the launch-pad and BCHH reference locations


In [ ]:
def get_unique_kml_location(
    dataframe: pd.DataFrame,
    kml_id: str,
) -> dict:
    matches = dataframe.loc[
        dataframe["kml_id"]
        .astype(str)
        .str.upper()
        .eq(kml_id.upper())
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one KML placemark {kml_id!r}; "
            f"found {len(matches)}."
        )

    row = matches.iloc[0].copy()

    # Normalize possible coordinate-column conventions.
    lat_key = (
        "lat"
        if "lat" in row.index
        else "latitude"
    )
    lon_key = (
        "lon"
        if "lon" in row.index
        else "longitude"
    )

    return {
        **row.to_dict(),
        "name": str(row.get("name", kml_id)),
        "lat": float(row[lat_key]),
        "lon": float(row[lon_key]),
    }


SLC40 = get_unique_kml_location(locations_df, "SLC40")
SLC41 = get_unique_kml_location(locations_df, "SLC41")

BCHH = get_sensor_location(
    bchh_sensors_df,
    sensor_name="Seismometer",
    display_name="BCHH",
)

print("SLC-40:", SLC40)
print("SLC-41:", SLC41)
print("BCHH:", BCHH)


## 5. Generate and save Figure 1

Geometry is fixed by the products above. Presentation changes should be made
in `figure1_utils.make_figure1()` or through its supported arguments, not by
redefining coordinates in this notebook.


In [ ]:
fig, axes = make_figure1(
    slc40=SLC40,
    slc41=SLC41,
    bchh=BCHH,
    bchh_sensors=bchh_sensors_df,
    geod=geod,
    ll_to_utm=ll_to_utm,
    utm_to_ll=utm_to_ll,
)

output_paths = save_figure(
    fig=fig,
    output_directory=FIGURE_DIR,
    filename_stem="fig01_slc40_bchh_location",
    extensions=("png", "pdf"),
    dpi=300,
)

print("Saved Figure 1:")
for path in output_paths:
    print(f"  {Path(path).resolve()}")

plt.show()


## Outputs

- `outputs/figures/fig01_slc40_bchh_location.png`
- `outputs/figures/fig01_slc40_bchh_location.pdf`

If only geometry metadata changes, rerun Notebook 02 and this notebook.
Instrument correction and downstream waveform measurements do not need to be
repeated.


# Part II — Key-event and catalogue figures


# Part I — Figure 1: deployment geometry


## 1. Imports and project paths

The notebook may be run from either the project root or the `notebooks/`
directory.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from pyproj import CRS, Geod, Transformer

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

MODULE_DIR = PROJECT_ROOT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from project_config import ensure_output_dirs
from geometry_products import read_geometry_products
from figure1_utils import (
    get_sensor_location,
    make_figure1,
    save_figure,
)

PATHS = ensure_output_dirs(PROJECT_ROOT)
DERIVED_DIR = PATHS["derived"]
FIGURE_DIR = PATHS["figures"]

WGS84 = CRS.from_epsg(4326)
UTM17N = CRS.from_epsg(32617)

geod = Geod(ellps="WGS84")
ll_to_utm = Transformer.from_crs(
    WGS84,
    UTM17N,
    always_xy=True,
)
utm_to_ll = Transformer.from_crs(
    UTM17N,
    WGS84,
    always_xy=True,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Derived data: {DERIVED_DIR}")
print(f"Figure output: {FIGURE_DIR}")


## 2. Load the standardized geometry products

Notebook 02 writes the event inventory and normalized channel, station, and
KML-location tables. Figure 1 consumes those products directly rather than
rereading StationXML or KML independently.


In [ ]:
(
    inventory_event,
    channels_df,
    stations_df,
    locations_df,
) = read_geometry_products(DERIVED_DIR)

print("Channel columns:")
print(channels_df.columns.tolist())

print("\nLocation columns:")
print(locations_df.columns.tolist())

display(stations_df)
display(channels_df)
display(locations_df)


## 3. Adapt the channel table for Figure 1

The standardized channel table uses descriptive column names such as
`latitude` and `longitude`. The existing Figure 1 utility expects the
concise plotting schema `sensor`, `label`, `lat`, `lon`, `easting`, and
`northing`. This cell creates those derived fields explicitly.

The three seismic components represent one physical seismometer location,
so they are collapsed to a single `Seismometer` row.


In [ ]:
CHANNEL_TO_SENSOR = {
    "DHZ": "Seismometer",
    "DHN": "Seismometer",
    "DHE": "Seismometer",
    "HHZ": "Seismometer",
    "HHN": "Seismometer",
    "HHE": "Seismometer",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
    "HD1": "HD1",
    "HD2": "HD2",
    "HD3": "HD3",
}

CHANNEL_TO_LABEL = {
    "DHZ": "BCHH",
    "DHN": "BCHH",
    "DHE": "BCHH",
    "HHZ": "BCHH",
    "HHN": "BCHH",
    "HHE": "BCHH",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
    "HD1": "HD1",
    "HD2": "HD2",
    "HD3": "HD3",
}

required_channel_columns = {
    "channel",
    "latitude",
    "longitude",
}
missing = required_channel_columns.difference(channels_df.columns)
if missing:
    raise KeyError(
        "The standardized channel table is missing required columns: "
        f"{sorted(missing)}"
    )

channels_for_map = channels_df.copy()
channel_codes = (
    channels_for_map["channel"]
    .astype(str)
    .str.upper()
)

channels_for_map["sensor"] = channel_codes.map(CHANNEL_TO_SENSOR)
channels_for_map["label"] = channel_codes.map(CHANNEL_TO_LABEL)
channels_for_map["lat"] = channels_for_map["latitude"].astype(float)
channels_for_map["lon"] = channels_for_map["longitude"].astype(float)

eastings, northings = ll_to_utm.transform(
    channels_for_map["lon"].to_numpy(),
    channels_for_map["lat"].to_numpy(),
)
channels_for_map["easting"] = eastings
channels_for_map["northing"] = northings

unmapped = channels_for_map.loc[
    channels_for_map["sensor"].isna(),
    ["seed_id", "channel", "sensor_description"],
]
if not unmapped.empty:
    print("Ignoring channels that are not used in Figure 1:")
    display(unmapped)

mapped = channels_for_map.dropna(subset=["sensor"]).copy()

seismometer_rows = mapped.loc[
    mapped["sensor"] == "Seismometer"
]
if seismometer_rows.empty:
    raise ValueError("No seismic component was mapped to 'Seismometer'.")

# All three components are co-located; retain one representative row.
seismometer_row = seismometer_rows.iloc[[0]].copy()

infrasound_rows = (
    mapped.loc[mapped["sensor"].isin(["HD1", "HD2", "HD3"])]
    .sort_values("sensor")
    .drop_duplicates(subset=["sensor"])
)

bchh_sensors_df = pd.concat(
    [seismometer_row, infrasound_rows],
    ignore_index=True,
)

expected_sensors = {"Seismometer", "HD1", "HD2", "HD3"}
actual_sensors = set(bchh_sensors_df["sensor"])
if actual_sensors != expected_sensors:
    raise ValueError(
        "Expected exactly these physical sensors: "
        f"{sorted(expected_sensors)}; found {sorted(actual_sensors)}"
    )

figure1_columns = [
    "sensor",
    "label",
    "lat",
    "lon",
    "easting",
    "northing",
    "channel",
    "seed_id",
    "sensor_description",
]
display(bchh_sensors_df[figure1_columns])


## 4. Extract the launch-pad and BCHH reference locations


In [ ]:
def get_unique_kml_location(
    dataframe: pd.DataFrame,
    kml_id: str,
) -> dict:
    matches = dataframe.loc[
        dataframe["kml_id"]
        .astype(str)
        .str.upper()
        .eq(kml_id.upper())
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one KML placemark {kml_id!r}; "
            f"found {len(matches)}."
        )

    row = matches.iloc[0].copy()

    # Normalize possible coordinate-column conventions.
    lat_key = (
        "lat"
        if "lat" in row.index
        else "latitude"
    )
    lon_key = (
        "lon"
        if "lon" in row.index
        else "longitude"
    )

    return {
        **row.to_dict(),
        "name": str(row.get("name", kml_id)),
        "lat": float(row[lat_key]),
        "lon": float(row[lon_key]),
    }


SLC40 = get_unique_kml_location(locations_df, "SLC40")
SLC41 = get_unique_kml_location(locations_df, "SLC41")

BCHH = get_sensor_location(
    bchh_sensors_df,
    sensor_name="Seismometer",
    display_name="BCHH",
)

print("SLC-40:", SLC40)
print("SLC-41:", SLC41)
print("BCHH:", BCHH)


## 5. Generate and save Figure 1

Geometry is fixed by the products above. Presentation changes should be made
in `figure1_utils.make_figure1()` or through its supported arguments, not by
redefining coordinates in this notebook.


In [ ]:
fig, axes = make_figure1(
    slc40=SLC40,
    slc41=SLC41,
    bchh=BCHH,
    bchh_sensors=bchh_sensors_df,
    geod=geod,
    ll_to_utm=ll_to_utm,
    utm_to_ll=utm_to_ll,
)

output_paths = save_figure(
    fig=fig,
    output_directory=FIGURE_DIR,
    filename_stem="fig01_slc40_bchh_location",
    extensions=("png", "pdf"),
    dpi=300,
)

print("Saved Figure 1:")
for path in output_paths:
    print(f"  {Path(path).resolve()}")

plt.show()


## Outputs

- `outputs/figures/fig01_slc40_bchh_location.png`
- `outputs/figures/fig01_slc40_bchh_location.pdf`

If only geometry metadata changes, rerun Notebook 02 and this notebook.
Instrument correction and downstream waveform measurements do not need to be
repeated.


# Part II — Key-event and catalogue figures


Read finalized products and generate publication figures. Scientific measurements are loaded from CSV rather than silently recalculated.


In [ ]:

from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
MODULE_DIR = PROJECT_ROOT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, UTCDateTime

from project_config import ensure_output_dirs

PATHS = ensure_output_dirs(PROJECT_ROOT)
DERIVED_DIR = PATHS["derived"]
FIGURE_DIR = PATHS["figures"]

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})


In [ ]:

from obspy import read
from plotting import plot_key_event_waveforms

st_corr = read(str(DERIVED_DIR / "bchh_corrected_analysis_window.pkl"),
               format="PICKLE")
measurements = pd.read_csv(
    DERIVED_DIR / "key_event_pressure_measurements.csv"
)


## Initial second-stage failure


In [ ]:

EXPLOSION_TIME = UTCDateTime("2016-09-01T13:07:12.080")
t0, t1 = EXPLOSION_TIME + 3.0 - 0.08, EXPLOSION_TIME + 5.0 - 0.08
st_event = st_corr.copy().trim(t0, t1)
event_results = measurements.loc[
    measurements["event"] == "Initial second-stage failure"
]
plot_key_event_waveforms(
    st_event,
    reference_time=t0,
    pressure_results=event_results,
    title="Initial second-stage failure",
    outfile=FIGURE_DIR / "second_stage_waveforms.png",
)
plt.show()


## Principal explosion


In [ ]:

t0, t1 = EXPLOSION_TIME + 6.0 - 0.08, EXPLOSION_TIME + 9.0 - 0.08
st_event = st_corr.copy().trim(t0, t1)
event_results = measurements.loc[
    measurements["event"] == "Principal explosion"
]
plot_key_event_waveforms(
    st_event,
    reference_time=t0,
    pressure_results=event_results,
    title="Principal Falcon 9 explosion",
    outfile=FIGURE_DIR / "principal_explosion_waveforms.png",
)
plt.show()


## Capsule sequence


In [ ]:

t0 = UTCDateTime("2016-09-01T13:07:27.0")
t1 = UTCDateTime("2016-09-01T13:07:30.0")
st_event = st_corr.copy().trim(t0, t1)
capsule_results = measurements.loc[
    measurements["event"].isin(["Capsule pulse 1", "Capsule pulse 2"])
]
# For a two-pulse figure, marker annotations should be added manually or by
# extending plotting.py to accept multiple measurement windows.
plot_key_event_waveforms(
    st_event,
    reference_time=t0,
    pressure_results=None,
    title="Capsule-related acoustic pulses",
    outfile=FIGURE_DIR / "capsule_waveforms.png",
    add_measurements=False,
)
plt.show()


## Array-result figures


In [ ]:

array_file = PROJECT_ROOT / "outputs" / "array_analysis_planar_refined" / "planar_array_results.csv"
if array_file.exists():
    array_results = pd.read_csv(array_file)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(
        np.arange(len(array_results)),
        array_results["best_back_azimuth_deg"],
        c=array_results["maximum_coherence_score"],
        s=22,
    )
    ax.set_xlabel("Event number")
    ax.set_ylabel("Back azimuth (degrees)")
    ax.set_title("Source direction through the explosion sequence")
    fig.savefig(
        FIGURE_DIR / "catalogue_back_azimuth.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
else:
    print("Run 05_planar_array_analysis.ipynb first.")
